# M-ViSER - Speech Emotion Recognition

**Repo**: https://github.com/Huu2412/M-ViSER

---
### Training Modes
| Stage | Mo ta | Lenh |
|---|---|---|
| **0** | End-to-End (Student + Teacher cung luc) | `--stage 0` |
| **1** | Chi train Teacher (Audio + Clean Text) | `--stage 1` |
| **2** | Student Distillation tu Teacher da freeze | `--stage 2 --teacher_ckpt ...` |

> **QUAN TRONG**: Chay tung cell theo thu tu 1 -> 2 -> 3 -> 4 -> 5x


In [ ]:
# ============================================================
# CELL 1: Clone repo (luon lay code moi nhat)
# ============================================================
import os

REPO_URL = 'https://github.com/Huu2412/M-ViSER.git'
REPO_DIR = '/kaggle/working/M-ViSER'

os.system(f'rm -rf {REPO_DIR}')
os.system(f'git clone {REPO_URL} {REPO_DIR}')

print('=== Latest 3 commits ===')
os.system(f'git -C {REPO_DIR} log --oneline -3')


In [ ]:
# ============================================================
# CELL 2: Fix torch CUDA + Cai dat dependencies
#
# Yeu cau: torch >= 2.6.0 (bat buoc do CVE-2025-32434)
# transformers moi nhat tu choi torch < 2.6 vi lo hong bao mat.
#
# Dung subprocess de test torch, tranh crash kernel khi torch bi hong.
# ============================================================
import subprocess, sys, os

MIN_TORCH = '2.6.0'

def pip(*args):
    r = subprocess.run([sys.executable, '-m', 'pip'] + list(args),
                       capture_output=True, text=True)
    return r.returncode, r.stderr

# --- Buoc 1: Test torch bang subprocess ---
print(f'=== Kiem tra torch (yeu cau >= {MIN_TORCH}) ===')
test_cmd = (
    'import torch; '
    'from packaging.version import Version; '
    'assert Version(torch.__version__.split("+")[0]) >= Version("2.6.0"), '
    '    f"torch {torch.__version__} < 2.6.0"; '
    'x = torch.tensor([1.0]).cuda(); '
    'torch.isfinite(x); '
    'print("TORCH_OK")'
)
r = subprocess.run(
    [sys.executable, '-c', test_cmd],
    capture_output=True, text=True, timeout=60
)
torch_ok = (r.returncode == 0 and 'TORCH_OK' in r.stdout)
print(f'Torch >= 2.6 + CUDA: {"OK" if torch_ok else "FAILED"}')
if not torch_ok:
    hint = (r.stderr + r.stdout)[-400:]
    print(f'  Hint: {hint}')

# --- Buoc 2: Neu can -> uninstall + reinstall torch >= 2.6 ---
if not torch_ok:
    print()
    print(f'==> Reinstalling torch >= {MIN_TORCH} cho Kaggle...')

    # Detect CUDA version tu nvidia-smi
    import re
    smi = subprocess.run(['nvidia-smi'], capture_output=True, text=True).stdout
    cuda_ver = '12.1'
    m = re.search(r'CUDA Version:\s*([\d.]+)', smi)
    if m:
        cuda_ver = m.group(1)
    print(f'  Detected CUDA: {cuda_ver}')

    major = int(cuda_ver.split('.')[0])
    # cu124 co torch 2.6+, cu121 cung co torch 2.6+
    cuda_tag = 'cu124' if major >= 12 else 'cu118'
    whl = f'https://download.pytorch.org/whl/{cuda_tag}'
    print(f'  Wheel index: {whl}')

    # Uninstall truoc
    print('  Uninstalling old torch and conflicting packages...')
    pip('uninstall', '-y', 'torch', 'torchaudio', 'torchvision', 'torchcodec')

    # Reinstall torch >= 2.6
    print(f'  Installing torch>={MIN_TORCH}...')
    rc, err = pip('install', '-q',
                  f'torch>={MIN_TORCH}', f'torchaudio>={MIN_TORCH}',
                  '--index-url', whl)

    if rc != 0:
        # Fallback: thu cu121
        print(f'  cu124 that bai, thu cu121...')
        pip('uninstall', '-y', 'torch', 'torchaudio', 'torchvision', 'torchcodec')
        rc, err = pip('install', '-q',
                      f'torch>={MIN_TORCH}', f'torchaudio>={MIN_TORCH}',
                      '--index-url', 'https://download.pytorch.org/whl/cu121')

    # Verify
    r2 = subprocess.run([sys.executable, '-c', test_cmd],
                        capture_output=True, text=True, timeout=60)
    if 'TORCH_OK' in r2.stdout:
        print()
        print('  Torch >= 2.6 da duoc cai thanh cong!')
        print()
        print('  *** RESTART KERNEL NGAY: Runtime -> Restart Session ***')
        print('  Sau do chay lai: Cell 1 -> Cell 2 -> Cell 3 -> ...')
    else:
        print(f'  Verify that bai: {(r2.stderr+r2.stdout)[-300:]}')
        print('  Thu xoa tay: pip uninstall torch torchaudio -y')
else:
    print('Torch OK, khong can reinstall.')

# --- Buoc 3: Cai cac thu vien khac (skip torch) ---
SKIP = {'torch', 'torchaudio', 'torchvision', 'torchcodec'}
req_path = f'{REPO_DIR}/requirements.txt'
with open(req_path) as f:
    lines = f.readlines()

pkgs = []
for line in lines:
    line = line.strip()
    if not line or line.startswith('#'):
        continue
    name = line.split('>=')[0].split('==')[0].split('<=')[0].strip().lower()
    if name not in SKIP:
        pkgs.append(line)

if pkgs:
    print(f'\nCai {len(pkgs)} packages khac...')
    rc, err = pip('install', '-q', *pkgs)
    
    # Force uninstall torchcodec to prevent HuggingFace datasets from trying to use it
    # which causes CUDA/ffmpeg linking errors on Kaggle with PyTorch 2.6
    pip('uninstall', '-y', 'torchcodec')
    
    if rc != 0:
        print(f'Canh bao: {err[-200:]}')
    print('Done.')


In [ ]:
# ============================================================
# CELL 3: Kiem tra GPU chi tiet
# (Neu Cell 2 bao RESTART KERNEL -> restart truoc khi chay cell nay)
# ============================================================
import torch, os

print(f'PyTorch version : {torch.__version__}')
print(f'CUDA available  : {torch.cuda.is_available()}')

if torch.cuda.is_available():
    props = torch.cuda.get_device_properties(0)
    print(f'GPU             : {torch.cuda.get_device_name(0)}')
    print(f'VRAM            : {props.total_memory/1e9:.1f} GB')
    print(f'CUDA version    : {torch.version.cuda}')
    print(f'Compute Cap.    : sm_{props.major}{props.minor}')

    try:
        _x = torch.tensor([1.0, float('nan')]).cuda()
        torch.isfinite(_x)
        del _x
        print('CUDA kernel test: PASSED')
    except Exception as e:
        print(f'CUDA kernel test: FAILED -> {e}')
        print('Quay lai Cell 2, sau do Restart Kernel!')
else:
    print('WARNING: Khong co GPU! Hay bat GPU trong Settings.')

# Kiem tra version >= 2.6
from packaging.version import Version
tv = Version(torch.__version__.split('+')[0])
if tv < Version('2.6.0'):
    print(f'WARNING: torch {torch.__version__} < 2.6.0 -> transformers se tu choi!')
    print('Hay chay lai Cell 2 de upgrade.')
else:
    print(f'Version check   : OK ({torch.__version__} >= 2.6.0)')

os.system('df -h /kaggle/working')


In [ ]:
# ============================================================
# CELL 4: Smoke Test (forward + backward pass)
# ============================================================
import sys, os
sys.path.insert(0, REPO_DIR)
os.chdir(REPO_DIR)

print(f'Working dir: {os.getcwd()}')
ret = os.system('CUDA_LAUNCH_BLOCKING=1 python smoke_test.py')
print('\nSmoke test: PASSED' if ret == 0 else '\nSmoke test: FAILED')


In [ ]:
# ============================================================
# CELL 5A: Train -- End-to-End (Stage 0)
# ============================================================
import sys, os
sys.path.insert(0, REPO_DIR)
os.chdir(REPO_DIR)
os.system('python train.py --config config/config.yaml --stage 0')


In [ ]:
# ============================================================
# CELL 5B: Train -- Stage 1: Teacher Only
# ============================================================
import sys, os
sys.path.insert(0, REPO_DIR)
os.chdir(REPO_DIR)
os.system('python train.py --config config/config.yaml --stage 1')


In [ ]:
# ============================================================
# CELL 5C: Train -- Stage 2: Student Distillation
#   Can chay Cell 5B truoc!
# ============================================================
import sys, os
sys.path.insert(0, REPO_DIR)
os.chdir(REPO_DIR)

TEACHER_CKPT = f'{REPO_DIR}/checkpoints/stage1_teacher/best_model.pt'
if not os.path.exists(TEACHER_CKPT):
    print(f'Teacher checkpoint khong tim thay: {TEACHER_CKPT}')
    print('  --> Hay chay Cell 5B (Stage 1) truoc!')
else:
    print(f'Teacher checkpoint: {TEACHER_CKPT}')
    os.system(
        f'python train.py --config config/config.yaml '
        f'--stage 2 --teacher_ckpt {TEACHER_CKPT}'
    )


In [ ]:
# ============================================================
# CELL 6: 5-Fold Cross-Validation
# ============================================================
import sys, os
sys.path.insert(0, REPO_DIR)
os.chdir(REPO_DIR)
FOLDS = '1 2 3 4 5'
os.system(f'python run_5fold.py --config config/config.yaml --folds {FOLDS}')


In [ ]:
# ============================================================
# CELL 7: Evaluate tren test set
# ============================================================
import sys, os
sys.path.insert(0, REPO_DIR)
os.chdir(REPO_DIR)

BEST_CKPT = f'{REPO_DIR}/checkpoints/best_model.pt'
for c in [
    f'{REPO_DIR}/checkpoints/stage2_student/best_model.pt',
    f'{REPO_DIR}/checkpoints/stage1_teacher/best_model.pt',
]:
    if not os.path.exists(BEST_CKPT) and os.path.exists(c):
        BEST_CKPT = c

print(f'Evaluating: {BEST_CKPT}')
os.system(f'python evaluate.py --config config/config.yaml --checkpoint {BEST_CKPT}')


In [ ]:
# ============================================================
# CELL 8: Nen va export checkpoint
# ============================================================
import os, shutil
from datetime import datetime

OUTPUT_DIR = '/kaggle/working'
CKPT_DIR   = f'{REPO_DIR}/checkpoints'
ts         = datetime.now().strftime('%Y%m%d_%H%M')
zip_base   = f'{OUTPUT_DIR}/mvisar_ckpt_{ts}'

if os.path.exists(CKPT_DIR):
    shutil.make_archive(zip_base, 'zip', CKPT_DIR)
    zip_file = zip_base + '.zip'
    size_mb  = os.path.getsize(zip_file) / 1e6
    print(f'Done: {zip_file} ({size_mb:.1f} MB)')
    print('Tai ve: Kaggle > Output tab')
else:
    print(f'Khong tim thay: {CKPT_DIR}')

print('\n=== Kaggle Output ===')
for fname in sorted(os.listdir(OUTPUT_DIR)):
    fpath = os.path.join(OUTPUT_DIR, fname)
    if os.path.isfile(fpath):
        print(f'  {fname}: {os.path.getsize(fpath)/1e6:.1f} MB')
